<a href="https://colab.research.google.com/github/sameerkarur/Data_science/blob/main/02_IITK_AIML_Core_Applied_Data_Science_with_Python/02_python_essentials/solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Essentials for DS — Solutions

Run setup first, then each solution cell.

## Setup

In [ ]:
from pathlib import Path
import os, sys, subprocess, urllib.request

# ⚡ Universal Colab & Workspace Setup
def setup_environment():
    if 'google.colab' in sys.modules or os.path.exists('/content'):
        colab_repo = Path('/content/Data_science')
        if not (colab_repo / 'datasets' / 'shared').exists():
            print("🚀 Google Colab detected: Cloning repository from GitHub to load shared datasets...")
            subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/sameerkarur/Data_science.git', str(colab_repo)], check=False)
        if colab_repo.exists():
            os.chdir(str(colab_repo))
            return colab_repo

    p = Path('.').resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'datasets' / 'shared').exists():
            return candidate
    return Path('.')

REPO_ROOT = setup_environment()
DATA_DIR = REPO_ROOT / 'datasets' / 'shared'

def ensure_dataset(filename):
    local_path = DATA_DIR / filename
    if not local_path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        raw_url = f"https://raw.githubusercontent.com/sameerkarur/Data_science/main/datasets/shared/{filename}"
        print(f"📥 Downloading {filename} from GitHub...")
        try:
            urllib.request.urlretrieve(raw_url, str(local_path))
        except Exception as e:
            print(f"Download failed: {e}")
    return local_path

print(f"Repo root : {REPO_ROOT}")
print(f"Datasets  : {DATA_DIR}")
if DATA_DIR.exists():
    print("Available CSVs:", sorted(p.name for p in DATA_DIR.glob('*.csv')))

import pandas as pd
import numpy as np

import pandas as pd
import numpy as np

sales_path = ensure_dataset('AusApparalSales4thQrt2020.csv')
df = pd.read_csv(sales_path)
print(df.shape)
df.head()


## Python for DS

**Q1.** Import numpy, pandas, matplotlib; print versions.

In [ ]:
import numpy as np, pandas as pd, matplotlib
print(np.__version__, pd.__version__, matplotlib.__version__)

**Q2.** Create ndarray 1..12 reshape (3,4).

In [ ]:
import numpy as np
a = np.arange(1, 13).reshape(3, 4)
print(a)

**Q3.** Vectorized: multiply sales array by 1.1.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
adj = df['Sales'].to_numpy() * 1.1
print(adj[:5])

**Q4.** Apply function to column with .apply(lambda).

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
df['Sales_k'] = df['Sales'].apply(lambda x: round(x/1000, 2))
print(df.head())

**Q5.** List comprehension filter Sales > 30000.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
high = [s for s in df['Sales'] if s > 30000]
print(len(high))

**Q6.** Dict comprehension state → total sales.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
m = {s: df.loc[df['State']==s, 'Sales'].sum() for s in df['State'].unique()}
print(list(m.items())[:3])

**Q7.** Use pathlib to list all CSV in DATA_DIR.

In [ ]:
csvs = sorted(DATA_DIR.glob('*.csv'))
print([p.name for p in csvs])

**Q8.** Read CSV with usecols to load only Sales, State.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv', usecols=['Sales','State'])
print(df.head())

**Q9.** Timing: %%timeit sum Python list vs numpy.

In [ ]:
import numpy as np
lst = list(range(10000))
arr = np.arange(10000)
print(sum(lst), arr.sum())

**Q10.** Use f-string format dataframe shape in message.

In [ ]:
df = pd.read_csv(DATA_DIR / 'marketing_data.csv')
print(f'Loaded {df.shape[0]} rows x {df.shape[1]} cols')

**Q11.** Try/except reading missing file gracefully.

In [ ]:
from pathlib import Path
try:
    pd.read_csv(DATA_DIR / 'missing.csv')
except FileNotFoundError as e:
    print('Not found:', e.filename if hasattr(e,'filename') else 'missing.csv')

**Q12.** Assert no negative Sales values.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
assert (df['Sales'] >= 0).all(), 'negative sales'
print('ok')

**Q13.** Use typing hint def top_n(series, n: int) -> pd.Series.

In [ ]:
def top_n(series: pd.Series, n: int) -> pd.Series:
    return series.sort_values(ascending=False).head(n)
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(top_n(df.groupby('State')['Sales'].sum(), 3))

**Q14.** Lambda in sort: sort states by name length.

In [ ]:
states = sorted(df['State'].unique(), key=lambda s: len(s)) if 'df' in dir() else ['NSW','VIC']
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(sorted(df['State'].unique(), key=len)[:5])

**Q15.** Map values: Group M→Men, W→Women if present.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df['Group'].map({'M': 'Men', 'W': 'Women'}).head())

**Q16.** Filter dataframe with boolean mask.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df[df['Sales'] > df['Sales'].median()].head())

**Q17.** Chain methods: groupby mean sort descending head.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df.groupby('State')['Sales'].mean().sort_values(ascending=False).head())

**Q18.** Use .pipe for custom function in chain.

In [ ]:
def add_ratio(d):
    d = d.copy(); d['ratio'] = d['Sales']/d['Unit']; return d
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(add_ratio(df).head())

**Q19.** Set pandas display options max 5 columns.

In [ ]:
pd.set_option('display.max_columns', 5)
df = pd.read_csv(DATA_DIR / 'HR_comma_sep.csv')
print(df.head())

**Q20.** Reset display options.

In [ ]:
pd.reset_option('display.max_columns')

**Q21.** Use random seed in numpy choice sample.

In [ ]:
import numpy as np
np.random.seed(42)
print(np.random.choice([1,2,3], size=5))

**Q22.** Bin Sales into quartiles with qcut.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
df['q'] = pd.qcut(df['Sales'], 4, labels=['Q1','Q2','Q3','Q4'])
print(df['q'].value_counts())

**Q23.** Cut Sales into custom bins.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
df['tier'] = pd.cut(df['Sales'], bins=[0,15000,25000,50000], labels=['L','M','H'])
print(df['tier'].value_counts())

**Q24.** Merge two small summary dataframes.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
a = df.groupby('State')['Sales'].sum().reset_index()
b = df.groupby('State')['Unit'].sum().reset_index()
print(a.merge(b, on='State').head())

**Q25.** Pivot table mean Sales by State and Group.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(pd.pivot_table(df, values='Sales', index='State', columns='Group', aggfunc='mean').head())

**Q26.** Melt wide to long (create wide first).

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
wide = df.groupby('State')['Sales'].sum().reset_index()
print(wide)

**Q27.** Use assign to add column inline.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df.assign(log_sales=lambda d: np.log1p(d['Sales'])).head())

**Q28.** Explode list column demo.

In [ ]:
df = pd.DataFrame({'k': ['a','b'], 'v': [[1,2],[3]]})
print(df.explode('v'))

**Q29.** String accessor: lower state names.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df['State'].str.lower().unique()[:5])

**Q30.** Extract regex pattern from column if string.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df['Date'].astype(str).str[:4].head())

**Q31.** Replace values with .replace dict.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df.replace({'M':'Male','W':'Female'}).head())

**Q32.** Drop duplicates subset State, Date.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(len(df), len(df.drop_duplicates(subset=['State','Date'])))

**Q33.** Fillna median for numeric column demo.

In [ ]:
df = pd.read_csv(DATA_DIR / 'HR_comma_sep.csv')
num = df.select_dtypes('number').columns[0]
df[num] = df[num].fillna(df[num].median())
print(df[num].isna().sum())

**Q34.** Clip Sales to percentile 1-99.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
lo, hi = df['Sales'].quantile([0.01, 0.99])
df['Sales_clip'] = df['Sales'].clip(lo, hi)
print(df['Sales_clip'].min(), df['Sales_clip'].max())

**Q35.** Use agg multiple functions.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df.groupby('State')['Sales'].agg(['mean','sum','count']).head())

**Q36.** Named aggregation in groupby.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df.groupby('State').agg(avg_sales=('Sales','mean'), total=('Sales','sum')).head())

**Q37.** Rolling mean window 7 on daily aggregated sales.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv', parse_dates=['Date'])
daily = df.groupby('Date')['Sales'].sum()
print(daily.rolling(7, min_periods=1).mean().head())

**Q38.** Shift for lag feature demo.

In [ ]:
daily = df.groupby('Date')['Sales'].sum() if 'df' in dir() else pd.Series([1,2,3])
print(daily.shift(1).head())

**Q39.** Cumulative sum sales over time.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv', parse_dates=['Date'])
daily = df.groupby('Date')['Sales'].sum().sort_index()
print(daily.cumsum().head())

**Q40.** Save figure to project folder.

In [ ]:
import matplotlib.pyplot as plt
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
plt.figure(); plt.hist(df['Sales'], bins=30)
fig_path = REPO_ROOT / 'Course_02_Applied_Data_Science/02_python_essentials/sales_hist.png'
plt.savefig(fig_path); plt.close()
print(fig_path.exists())

**Q41.** Use os.environ.get for optional config.

In [ ]:
import os
DATA_PATH = os.environ.get('AIML_DATA', str(DATA_DIR))
print(DATA_PATH)

**Q42.** Write reusable load_sales() function.

In [ ]:
def load_sales():
    return pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(load_sales().shape)

**Q43.** Profile memory usage .memory_usage(deep=True).

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
print(df.memory_usage(deep=True).sum())

**Q44.** Convert column to category dtype.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
df['State'] = df['State'].astype('category')
print(df['State'].dtype)

**Q45.** Use pd.to_numeric errors='coerce'.

In [ ]:
s = pd.Series(['1','2','x'])
print(pd.to_numeric(s, errors='coerce'))

**Q46.** Build pytest-style assert for mean > 0.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv')
assert df['Sales'].mean() > 0

**Q47.** Docstring module-level practice function.

In [ ]:
def summarize(df):
    """Return shape and null counts."""
    return df.shape, df.isna().sum().sum()
print(summarize(pd.read_csv(DATA_DIR / 'loan_data.csv')))

**Q48.** Use __name__ == '__main__' guard pattern.

In [ ]:
def main():
    print('run as script')
if __name__ == '__main__':
    main()

**Q49.** Create requirements check list programmatically.

In [ ]:
needed = ['numpy','pandas','matplotlib','seaborn','sklearn']
import importlib
print({m: bool(importlib.util.find_spec(m)) for m in needed})

**Q50.** Use pd.read_csv with parse_dates and infer_datetime_format.

In [ ]:
df = pd.read_csv(DATA_DIR / 'AusApparalSales4thQrt2020.csv', parse_dates=['Date'])
print(df['Date'].dtype)